# 第十六课:Top-K——砍掉长尾

讲义:[16-top-k.md](16-top-k.md)。

In [ ]:
import numpy as np

def softmax(x):
    m = np.max(x)
    e = np.exp(x - m)
    return e / e.sum()

def top_k(logits, k):
    idx = np.argsort(-logits)[:k]
    mask = np.full_like(logits, -np.inf)
    mask[idx] = logits[idx]
    return mask

logits = np.array([1.0, 4.0, 2.0, 0.0])
print('原始分布:', np.round(softmax(logits), 4))
print('top-2 后 :', np.round(softmax(top_k(logits, 2)), 4))
print('top-1 后 :', np.round(softmax(top_k(logits, 1)), 4), '← K=1 = greedy')

## argpartition 高效版(作业 C2)

In [ ]:
def top_k_fast(logits, k):
    part = np.argpartition(-logits, k - 1)[:k]     # O(n) 粗选前 k
    idx = part[np.argsort(-logits[part])]          # 仅对 k 个排序
    mask = np.full_like(logits, -np.inf)
    mask[idx] = logits[idx]
    return mask

print('两版结果一致:', np.allclose(top_k(logits, 2), top_k_fast(logits, 2)))

import time
big = np.random.default_rng(0).normal(size=151_936)
for name, fn in [('argsort 版', top_k), ('argpartition 版', top_k_fast)]:
    t0 = time.perf_counter()
    for _ in range(100):
        fn(big, 100)
    print(f'{name}: {(time.perf_counter()-t0)*10:.2f} ms/次')

In [ ]:
# TODO(C3): 完整采样 sample_top_k(logits, k, rng, T=1.0)
# TODO(D1): 2048 token 生成的截断开销估算
# TODO(D3): 「手动重归一」与「-inf 掩码」在带温度时的差异实验